# Notebook 03 — Treinamento dos Modelos

**Propósito:** treinar Naive Sazonal, ETS, SARIMA e Prophet nas seis séries (3 municípios × 2 tributos), produzir previsões via rolling origin CV, salvar resultados.

**Tempo esperado:** ~30–60 minutos. Sem LSTM, sem GPU.

In [ ]:
from forecasting.config import load_config
from forecasting.plotting import setup_matplotlib_thesis
from forecasting import io as fio, models

setup_matplotlib_thesis()
models.set_global_seeds(42)
cfg = load_config()
monthly = fio.load_monthly_series(cfg)

In [ ]:
# Loop principal: (municipio, tributo, modelo, horizonte).
fit_functions = {
    'naive_seasonal': models.fit_naive_seasonal,
    'ets':            models.fit_ets,
    'sarima':         models.fit_sarima,
    'prophet':        models.fit_prophet,
}

for mun_key, mun in cfg.municipalities.items():
    for tributo in cfg.tributos:
        # serie = monthly[(monthly.cod_ibge == mun.cod_ibge)][tributo.lower()]
        for modelo, fit_fn in fit_functions.items():
            cv = models.rolling_origin_cv(
                series=None,  # passar `serie` real aqui
                fit_fn=fit_fn,
                initial_window=72,
                horizons=cfg.horizons,
            )
            for h, predictions in cv.predictions.items():
                fio.save_forecast(predictions, cfg, mun_key, tributo, modelo, h)

In [ ]:
# Geracao de tabelas de hiperparametros e figuras de previsao.
artifacts = models.run_all(cfg)
for a in artifacts:
    print(f'  + {a}')